## Detection & Segmentation — From Scratch Implementation

This notebook implements **YOLO**, **Mask R-CNN**, and **U-Net** architectures
**from scratch** (no pretrained weights) for clothing detection and segmentation.
- Subset: **1000 training images**, full validation set
- Training: **1 epoch** per model

In [1]:
!pip install pycocotools opencv-python matplotlib timm ultralytics segmentation-models-pytorch torchmetrics scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.9 MB/s eta 0:00:00


In [2]:
import os, json, cv2, yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter

os.chdir('/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd')

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 512
NUM_CLASSES = 5
SUBSET_SIZE = 1000
EPOCHS      = 1

TRAIN_IMG  = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/image"
TRAIN_ANNO = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/annos"
VAL_IMG    = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/image"
VAL_ANNO   = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/annos"

train_df = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/train_labels_top5_new.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/val_labels_top5_new.csv")

In [3]:
train_anno_dir = "./train/train/train/annos"
category_counter = Counter()
for file in tqdm(os.listdir(train_anno_dir)):
    if not file.endswith(".json"): continue
    with open(os.path.join(train_anno_dir, file)) as f:
        data = json.load(f)
    for key in data:
        if "item" not in key: continue
        category = data[key]["category_name"]
        category_counter[category] += 1

TOP5 = [cat for cat, _ in category_counter.most_common(5)]
label_map = {cat: i for i, cat in enumerate(TOP5)}
print("Top 5 categories:", TOP5)
print("Label mapping:", label_map)

100%|██████████| 191961/191961 [27:44<00:00, 115.32it/s]

Top 5 categories: ['short sleeve top', 'trousers', 'shorts', 'long sleeve top', 'skirt']
Label mapping: {'short sleeve top': 0, 'trousers': 1, 'shorts': 2, 'long sleeve top': 3, 'skirt': 4}


In [4]:
def parse_anno(img_name, anno_dir, label_map):
    path = os.path.join(anno_dir, img_name.replace(".jpg", ".json"))
    items = []
    if not os.path.exists(path): return items
    with open(path) as f:
        data = json.load(f)
    for key in data:
        if "item" not in key: continue
        item = data[key]
        cat  = item.get("category_name", "")
        if cat not in label_map: continue
        bbox = item.get("bounding_box", None)
        segs = item.get("segmentation", [])
        if bbox is None or not segs: continue
        items.append({"label": label_map[cat], "bbox": bbox, "segmentation": segs})
    return items

def poly_to_mask(segmentation, H, W):
    mask = np.zeros((H, W), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [pts], 1)
    return mask

def collate_fn(batch):
    return tuple(zip(*batch))

In [5]:
# Subset the training data to 1000 images
train_df_subset = train_df.sample(n=min(SUBSET_SIZE, len(train_df)), random_state=42).reset_index(drop=True)
print(f"Training subset: {len(train_df_subset)} images")
print(f"Validation set:  {len(val_df)} images")

Training subset: 1000 images
Validation set:  32153 images


### 1. Mask R-CNN — From Scratch

**Architecture:** Custom ResNet-50 backbone (randomly initialized, no pretrained weights) → FPN → RPN → ROI Heads (classification + bbox regression + mask head).

In [8]:
import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection import MaskRCNN
from torchvision.models.resnet import ResNet, Bottleneck
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

class MRCNNDataset(Dataset):
    def __init__(self, df, img_dir, anno_dir, size=IMG_SIZE):
        self.df = df; self.img_dir = img_dir
        self.anno_dir = anno_dir; self.size = size

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        W, H = img.size
        img  = img.resize((self.size, self.size))
        img_t = TF.to_tensor(img)
        sx, sy = self.size / W, self.size / H

        boxes, labels, masks = [], [], []
        for ann in parse_anno(img_name, self.anno_dir, label_map):
            x1, y1, x2, y2 = [c * s for c, s in zip(ann["bbox"], [sx, sy, sx, sy])]
            if x2 <= x1 or y2 <= y1:
                continue
            sc = []
            for poly in ann["segmentation"]:
                # fix: scale as float, then cast to int32
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                pts[:, 0] *= sx; pts[:, 1] *= sy
                sc.append(pts.astype(np.int32).flatten().tolist())
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["label"] + 1)
            masks.append(poly_to_mask(sc, self.size, self.size))

        if len(boxes) == 0:
            t = {"boxes":  torch.zeros((0, 4), dtype=torch.float32),
                 "labels": torch.zeros((0,),   dtype=torch.int64),
                 "masks":  torch.zeros((0, self.size, self.size), dtype=torch.uint8)}
        else:
            t = {"boxes":  torch.tensor(boxes,          dtype=torch.float32),
                 "labels": torch.tensor(labels,         dtype=torch.int64),
                 "masks":  torch.tensor(np.stack(masks), dtype=torch.uint8)}
        t["image_id"] = torch.tensor([idx])
        return img_t, t

train_mrcnn_dl = DataLoader(MRCNNDataset(train_df_subset, TRAIN_IMG, TRAIN_ANNO),
                             batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_mrcnn_dl   = DataLoader(MRCNNDataset(val_df,   VAL_IMG,   VAL_ANNO),
                             batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

In [10]:
def get_maskrcnn_scratch():
    # ResNet-50 backbone with RANDOM weights (from scratch, no pretrained)
    backbone_net = ResNet(Bottleneck, [3, 4, 6, 3])   # random init

    # return_layers maps ResNet children to FPN level names
    return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
    in_channels_list = [256, 512, 1024, 2048]

    backbone_fpn = BackboneWithFPN(
        backbone_net,
        return_layers=return_layers,
        in_channels_list=in_channels_list,
        out_channels=256
    )

    # Anchor generator
    anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)

    # ROI pooling
    roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"], output_size=7, sampling_ratio=2
    )
    mask_roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"], output_size=14, sampling_ratio=2
    )

    model = MaskRCNN(
        backbone_fpn,
        num_classes=NUM_CLASSES + 1,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        mask_roi_pool=mask_roi_pooler,
    )
    return model

In [11]:
maskrcnn  = get_maskrcnn_scratch().to(DEVICE)
opt_mrcnn = torch.optim.Adam([p for p in maskrcnn.parameters() if p.requires_grad], lr=1e-4)

print("Mask R-CNN (from scratch) params:", f"{sum(p.numel() for p in maskrcnn.parameters())/1e6:.1f}M")

best_mrcnn_loss = float("inf")

for epoch in range(EPOCHS):
    maskrcnn.train()
    total = 0
    for imgs, targets in tqdm(train_mrcnn_dl, desc=f"MRCNN Epoch {epoch+1}"):
        imgs    = [i.to(DEVICE) for i in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss    = sum(maskrcnn(imgs, targets).values())
        opt_mrcnn.zero_grad(); loss.backward(); opt_mrcnn.step()
        total  += loss.item()
    avg = total / len(train_mrcnn_dl)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.4f}")
    if avg < best_mrcnn_loss:
        best_mrcnn_loss = avg
        torch.save(maskrcnn.state_dict(), "/kaggle/working/best_maskrcnn_scratch.pth")

Mask R-CNN (from scratch) params: 44.0M


MRCNN Epoch 1: 100%|██████████| 250/250 [04:09<00:00,  1.00it/s]


Epoch 1/1 | Loss: 1.0265


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

maskrcnn.load_state_dict(torch.load("/kaggle/working/best_maskrcnn_scratch.pth"))
maskrcnn.eval()

metric_mrcnn  = MeanAveragePrecision(iou_type="segm")
iou_cls_mrcnn = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for imgs, targets in tqdm(val_mrcnn_dl, desc="MRCNN Val"):
        imgs  = [i.to(DEVICE) for i in imgs]
        preds = maskrcnn(imgs)

        # threshold float masks → uint8 before metric update
        preds_fixed = []
        for p in preds:
            preds_fixed.append({
                "boxes":  p["boxes"].cpu(),
                "scores": p["scores"].cpu(),
                "labels": p["labels"].cpu(),
                "masks":  (p["masks"].squeeze(1).cpu() > 0.5).to(torch.uint8)
            })

        targets_fixed = [{k: v.cpu() for k, v in t.items()} for t in targets]

        metric_mrcnn.update(preds_fixed, targets_fixed)

        for pred, tgt in zip(preds_fixed, targets_fixed):
            for pm, pl in zip(pred["masks"], pred["labels"]):
                pb = pm.numpy()
                c  = pl.item() - 1
                if not (0 <= c < NUM_CLASSES): continue
                gt = tgt["masks"][tgt["labels"] == (c + 1)].numpy()
                if len(gt) == 0: continue
                iou_cls_mrcnn[c].append(
                    max((pb & g).sum() / ((pb | g).sum() + 1e-6) for g in gt))

res_mrcnn     = metric_mrcnn.compute()
per_iou_mrcnn = [np.mean(v) if v else 0.0 for v in iou_cls_mrcnn]
miou_mrcnn    = np.mean(per_iou_mrcnn)
dice_mrcnn    = np.mean([2 * iou / (1 + iou) for iou in per_iou_mrcnn])

MRCNN Val:   8%|▊         | 648/8039 [07:28<1:31:40,  1.34it/s]

In [ ]:
print("Mask R-CNN (From Scratch) Results")
print(f"mAP@[0.5:0.95] (segm) : {res_mrcnn['map'].item():.4f}")
print(f"mAP@0.50       (segm) : {res_mrcnn['map_50'].item():.4f}")
print(f"mIoU                  : {miou_mrcnn:.4f}")
print(f"Dice (macro)          : {dice_mrcnn:.4f}")
print(f"Per-class IoU         : {[f'{v:.3f}' for v in per_iou_mrcnn]}")

### 2. YOLO — From Scratch

**Architecture:** YOLOv8s-seg built from YAML config (CSPDarknet backbone + FPN/PAN neck + detection & segmentation heads). All weights randomly initialized — `yolov8s-seg.yaml` instead of `yolov8s-seg.pt`.

In [ ]:
import yaml
from ultralytics import YOLO

YOLO_DIR = "/kaggle/working/yolo_data_scratch"


def build_yolo_split(df, img_dir, anno_dir, split):
    os.makedirs(f"{YOLO_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DIR}/labels/{split}", exist_ok=True)
    for _, row in df.iterrows():
        name = row["image"]
        src  = os.path.join(img_dir, name)
        if not os.path.exists(src): continue
        img  = cv2.imread(src)
        H, W = img.shape[:2]
        stem = os.path.splitext(name)[0]
        cv2.imwrite(f"{YOLO_DIR}/images/{split}/{stem}.jpg", img)
        lines = []
        for ann in parse_anno(name, anno_dir, label_map):
            cls, pts_flat = ann["label"], []
            for poly in ann["segmentation"]:
                arr = np.array(poly).reshape(-1,2).astype(float)
                arr[:,0] /= W; arr[:,1] /= H
                pts_flat.extend(arr.flatten().tolist())
            if pts_flat:
                lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in pts_flat))
        with open(f"{YOLO_DIR}/labels/{split}/{stem}.txt","w") as f:
            f.write("\n".join(lines))

build_yolo_split(train_df_subset, TRAIN_IMG, TRAIN_ANNO, "train")
build_yolo_split(val_df,   VAL_IMG,   VAL_ANNO,   "val")

yaml.dump({"path": YOLO_DIR, "train": "images/train", "val": "images/val",
           "nc": NUM_CLASSES, "names": list(label_map.keys())},
          open(f"{YOLO_DIR}/data.yaml","w"))
print("YOLO dataset ready (from scratch subset)")

In [ ]:
from ultralytics import YOLO

# .yaml => from-scratch (random init),  .pt => pretrained
yolo_model = YOLO("yolov8s-seg.yaml")

yolo_model.train(
    data    = f"{YOLO_DIR}/data.yaml",
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = 8,
    device  = 0 if torch.cuda.is_available() else "cpu",
    project = "/kaggle/working",
    name    = "yolo_seg_scratch",
    exist_ok = True,
    pretrained = False,
)

In [ ]:
val_res_yolo = yolo_model.val(data=f"{YOLO_DIR}/data.yaml")

map_box_yolo = val_res_yolo.box.map
map_seg_yolo = val_res_yolo.seg.map

print("\n=== YOLO (From Scratch) Results ===")
print(f"Box  mAP@[0.5:0.95] : {map_box_yolo:.4f}")
print(f"Seg  mAP@[0.5:0.95] : {map_seg_yolo:.4f}")
print(f"Box  mAP@0.50       : {val_res_yolo.box.map50:.4f}")
print(f"Seg  mAP@0.50       : {val_res_yolo.seg.map50:.4f}")

### 3. U-Net — From Scratch

**Architecture:** Custom CNN encoder (DoubleConv blocks) that progressively downsamples → symmetric decoder with skip connections → pixel-wise segmentation output. Post-processing: connected component analysis for instance extraction + bounding box derivation.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNetScratch(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)

        # Encoder path
        for f in features:
            self.downs.append(DoubleConv(in_channels, f))
            in_channels = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder path
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f*2, f))

        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skip_connections[idx // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat((skip, x), dim=1)
            x = self.ups[idx + 1](x)
        return self.final_conv(x)

In [ ]:
class UNetDataset(Dataset):
    def __init__(self, df, img_dir, anno_dir, size=IMG_SIZE):
        self.df = df; self.img_dir = img_dir
        self.anno_dir = anno_dir; self.size = size

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        W, H = img.size
        img  = img.resize((self.size, self.size))
        img_t = TF.to_tensor(img)
        sx, sy = self.size/W, self.size/H

        seg_mask = np.zeros((self.size, self.size), dtype=np.int64)
        for ann in parse_anno(img_name, self.anno_dir, label_map):
            sc = []
            for poly in ann["segmentation"]:
                pts = np.array(poly).reshape(-1,2)
                pts[:,0] *= sx; pts[:,1] *= sy
                sc.append(pts.flatten().tolist())
            m = poly_to_mask(sc, self.size, self.size)
            seg_mask[m == 1] = ann["label"] + 1
        return img_t, torch.tensor(seg_mask, dtype=torch.long)

train_unet_dl = DataLoader(UNetDataset(train_df_subset, TRAIN_IMG, TRAIN_ANNO),
                            batch_size=8, shuffle=True,  num_workers=2)
val_unet_dl   = DataLoader(UNetDataset(val_df,   VAL_IMG,   VAL_ANNO),
                            batch_size=8, shuffle=False, num_workers=2)

In [ ]:
unet = UNetScratch(in_channels=3, num_classes=NUM_CLASSES+1).to(DEVICE)
ce_loss  = nn.CrossEntropyLoss()
opt_unet = torch.optim.Adam(unet.parameters(), lr=1e-4)

print("U-Net (from scratch) params:", f"{sum(p.numel() for p in unet.parameters())/1e6:.1f}M")

best_unet_loss = float("inf")

for epoch in range(EPOCHS):
    unet.train()
    total = 0
    for imgs, masks in tqdm(train_unet_dl, desc=f"UNet Epoch {epoch+1}"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        loss = ce_loss(unet(imgs), masks)
        opt_unet.zero_grad(); loss.backward(); opt_unet.step()
        total += loss.item()
    avg = total / len(train_unet_dl)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.4f}")
    if avg < best_unet_loss:
        best_unet_loss = avg
        torch.save(unet.state_dict(), "/kaggle/working/best_unet_scratch.pth")

In [ ]:
from scipy import ndimage

unet.load_state_dict(torch.load("/kaggle/working/best_unet_scratch.pth"))
unet.eval()

iou_cls_unet  = [[] for _ in range(NUM_CLASSES)]
dice_cls_unet = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for imgs, masks in tqdm(val_unet_dl, desc="UNet Val"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = unet(imgs).argmax(dim=1)
        for c in range(NUM_CLASSES):
            p = (preds == c+1).cpu().numpy()
            g = (masks == c+1).cpu().numpy()
            inter = (p & g).sum(); union = (p | g).sum()
            if union > 0:
                iou_cls_unet[c].append(inter / (union + 1e-6))
                dice_cls_unet[c].append(2*inter / (p.sum() + g.sum() + 1e-6))

per_iou_unet  = [np.mean(v) if v else 0.0 for v in iou_cls_unet]
per_dice_unet = [np.mean(v) if v else 0.0 for v in dice_cls_unet]
miou_unet = np.mean(per_iou_unet)
dice_unet = np.mean(per_dice_unet)

print("\n=== U-Net (From Scratch) Results ===")
print(f"mIoU         : {miou_unet:.4f}")
print(f"Dice (macro) : {dice_unet:.4f}")
print(f"Per-class IoU: {[f'{v:.3f}' for v in per_iou_unet]}")

# Instance post-processing via connected components
def unet_instances(pred_mask):
    instances = []
    for c in range(NUM_CLASSES):
        binary  = (pred_mask == c+1).astype(np.uint8)
        labeled, n = ndimage.label(binary)
        for i in range(1, n+1):
            comp = (labeled == i)
            ys, xs = np.where(comp)
            if len(xs) < 10: continue
            instances.append({
                "bbox":  [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
                "mask":  comp.astype(np.uint8), "label": c})
    return instances

# Demo instance extraction
with torch.no_grad():
    sample_img, _ = next(iter(val_unet_dl))
    sample_pred = unet(sample_img.to(DEVICE)).argmax(dim=1)[0].cpu().numpy()
    instances = unet_instances(sample_pred)
    print(f"Instance extraction demo: found {len(instances)} instances in sample image")

### 4. Summary Table — All From-Scratch Models

In [ ]:
summary = pd.DataFrame({
    "Model"            : ["Mask R-CNN (scratch)", "YOLO (scratch)", "U-Net (scratch)"],
    "mAP@[0.5:0.95]"   : [f"{res_mrcnn['map'].item():.4f}", f"{map_seg_yolo:.4f}", "N/A"],
    "mIoU"             : [f"{miou_mrcnn:.4f}",  "—",  f"{miou_unet:.4f}"],
    "Dice (macro)"     : [f"{dice_mrcnn:.4f}",  "—",  f"{dice_unet:.4f}"],
})
print(summary.to_string(index=False))